In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/train.csv")
print(df.shape)    # expect (891, 12)

(891, 12)


In [2]:
print(df["Survived"].mean())                  # about 0.38
print(df.groupby("Sex")["Survived"].mean())   # female ~0.74, male ~0.19
print(df.isna().sum())                        # Age, Cabin, Embarked have blanks

0.3838383838383838
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(712, 7) (179, 7)
0.38342696629213485 0.3854748603351955


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

num_cols = ["Age", "Fare", "SibSp", "Parch"]
cat_cols = ["Sex", "Pclass", "Embarked"]

num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

prep = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])

logreg = Pipeline([
    ("prep", prep),
    ("model", LogisticRegression(max_iter=1000)),
])

scores = cross_val_score(logreg, X_train, y_train, cv=5, scoring="roc_auc")
print("ROC-AUC per fold:", scores.round(3))
print("Mean:", scores.mean().round(3))

ROC-AUC per fold: [0.834 0.819 0.875 0.874 0.872]
Mean: 0.855


In [5]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
rows = []

for name, clf in models.items():
    pipe = Pipeline([("prep", prep), ("model", clf)])
    cv = cross_validate(pipe, X_train, y_train, cv=5,
                        scoring=scoring, return_train_score=True)
    rows.append({
        "Model": name,
        "Accuracy": cv["test_accuracy"].mean(),
        "Precision": cv["test_precision"].mean(),
        "Recall": cv["test_recall"].mean(),
        "F1": cv["test_f1"].mean(),
        "Val ROC-AUC": cv["test_roc_auc"].mean(),
        "Train ROC-AUC": cv["train_roc_auc"].mean(),
    })

results = pd.DataFrame(rows).set_index("Model").round(3)
results

,Accuracy,Precision,Recall,F1,Val ROC-AUC,Train ROC-AUC
Model,,,,,,
Logistic Regression,0.798,0.756,0.700,0.724,0.855,0.860
Decision Tree,0.794,0.738,0.715,0.725,0.782,0.999
Random Forest,0.791,0.732,0.715,0.723,0.851,0.998


In [6]:
from sklearn.dummy import DummyClassifier

models = {
    "Baseline (always 'died')": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree (depth 4)": DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=10, random_state=42),
    "Random Forest (depth 6)": RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42),
}

rows = []
for name, clf in models.items():
    pipe = Pipeline([("prep", prep), ("model", clf)])
    cv = cross_validate(pipe, X_train, y_train, cv=5,
                        scoring=["accuracy", "f1", "roc_auc"],
                        return_train_score=True)
    rows.append({
        "Model": name,
        "Accuracy": cv["test_accuracy"].mean(),
        "F1": cv["test_f1"].mean(),
        "Val AUC": cv["test_roc_auc"].mean(),
        "Val AUC std": cv["test_roc_auc"].std(),
        "Train AUC": cv["train_roc_auc"].mean(),
    })

pd.DataFrame(rows).set_index("Model").round(3)

,Accuracy,F1,Val AUC,Val AUC std,Train AUC
Model,,,,,
Baseline (always 'died'),0.617,0.000,0.500,0.000,0.500
Logistic Regression,0.798,0.724,0.855,0.023,0.860
Decision Tree (depth 4),0.815,0.732,0.847,0.014,0.883
Random Forest (depth 6),0.823,0.738,0.870,0.018,0.921


In [7]:
def add_features(d):
    d = d.copy()
    d["Title"] = d["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
    d["Title"] = d["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    d["Title"] = d["Title"].where(d["Title"].isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
    d["FamilySize"] = d["SibSp"] + d["Parch"] + 1
    d["IsAlone"] = (d["FamilySize"] == 1).astype(int)
    d["HasCabin"] = d["Cabin"].notna().astype(int)
    return d

df2 = add_features(df)
X2 = df2.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"])
y2 = df2["Survived"]

# same random_state and stratify, so the same passengers land in train and test
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=42
)

num2 = ["Age", "Fare", "SibSp", "Parch", "FamilySize"]
cat2 = ["Sex", "Pclass", "Embarked", "Title"]
bin2 = ["IsAlone", "HasCabin"]

prep2 = ColumnTransformer([
    ("num", num_pipe, num2),
    ("cat", cat_pipe, cat2),
    ("bin", "passthrough", bin2),
])

models2 = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree (depth 4)": DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=10, random_state=42),
    "Random Forest (depth 6)": RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42),
}

rows = []
for name, clf in models2.items():
    pipe = Pipeline([("prep", prep2), ("model", clf)])
    cv = cross_validate(pipe, X2_train, y2_train, cv=5,
                        scoring=["accuracy", "f1", "roc_auc"],
                        return_train_score=True)
    rows.append({
        "Model": name,
        "Accuracy": cv["test_accuracy"].mean(),
        "F1": cv["test_f1"].mean(),
        "Val AUC": cv["test_roc_auc"].mean(),
        "Val AUC std": cv["test_roc_auc"].std(),
        "Train AUC": cv["train_roc_auc"].mean(),
    })

pd.DataFrame(rows).set_index("Model").round(3)

,Accuracy,F1,Val AUC,Val AUC std,Train AUC
Model,,,,,
Logistic Regression,0.823,0.763,0.869,0.014,0.879
Decision Tree (depth 4),0.791,0.709,0.838,0.038,0.891
Random Forest (depth 6),0.819,0.743,0.869,0.016,0.921


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor

known = df2[df2["Age"].notna()]
Xr = known[["Pclass", "Sex", "SibSp", "Parch", "Fare", "Embarked", "Title"]]
yr = known["Age"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)

num_r = ["SibSp", "Parch", "Fare"]
cat_r = ["Sex", "Pclass", "Embarked", "Title"]
prep_r = ColumnTransformer([
    ("num", num_pipe, num_r),
    ("cat", cat_pipe, cat_r),
])

reg_models = {
    "Baseline (predict mean age)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Decision Tree (depth 4)": DecisionTreeRegressor(
        max_depth=4, min_samples_leaf=10, random_state=42),
    "Random Forest (depth 6)": RandomForestRegressor(
        n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42),
}

rows = []
for name, reg in reg_models.items():
    pipe = Pipeline([("prep", prep_r), ("model", reg)])
    cv = cross_validate(pipe, Xr_train, yr_train, cv=5,
                        scoring=["neg_root_mean_squared_error", "r2"],
                        return_train_score=True)
    rows.append({
        "Model": name,
        "Val RMSE (years)": -cv["test_neg_root_mean_squared_error"].mean(),
        "Val RMSE std": cv["test_neg_root_mean_squared_error"].std(),
        "Train RMSE": -cv["train_neg_root_mean_squared_error"].mean(),
        "Val R2": cv["test_r2"].mean(),
    })

pd.DataFrame(rows).set_index("Model").round(2)

,Val RMSE (years),Val RMSE std,Train RMSE,Val R2
Model,,,,
Baseline (predict mean age),14.76,0.74,14.71,-0.02
Linear Regression,11.47,0.83,11.14,0.38
Decision Tree (depth 4),11.20,0.95,10.83,0.41
Random Forest (depth 6),11.33,0.99,10.04,0.40


In [9]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

best = Pipeline([("prep", prep2), ("model", LogisticRegression(max_iter=1000))])
pred = cross_val_predict(best, X2_train, y2_train, cv=5)

print(confusion_matrix(y2_train, pred))
print(classification_report(y2_train, pred, target_names=["Died", "Survived"]))

err = X2_train.copy()
err["actual"] = y2_train
err["pred"] = pred
err["wrong"] = (err["actual"] != err["pred"]).astype(int)
err["AgeGroup"] = pd.cut(err["Age"], [0, 12, 18, 35, 60, 100],
                         labels=["child", "teen", "young adult", "adult", "senior"])
err["AgeGroup"] = err["AgeGroup"].cat.add_categories("missing").fillna("missing")

for col in ["Sex", "Pclass", "Title", "AgeGroup", "IsAlone"]:
    print(err.groupby(col, observed=True)["wrong"].agg(["mean", "count"]).round(3), "\n")

wrong_rows = err[err["wrong"] == 1].copy()
wrong_rows["Name"] = df.loc[wrong_rows.index, "Name"]
wrong_rows[["Name", "Sex", "Age", "Pclass", "actual", "pred"]].head(15)

[[383  56]
 [ 70 203]]
              precision    recall  f1-score   support

        Died       0.85      0.87      0.86       439
    Survived       0.78      0.74      0.76       273

    accuracy                           0.82       712
   macro avg       0.81      0.81      0.81       712
weighted avg       0.82      0.82      0.82       712

         mean  count
Sex                 
female  0.202    253
male    0.163    459 

         mean  count
Pclass              
1       0.216    171
2       0.080    150
3       0.197    391 

         mean  count
Title               
Master  0.129     31
Miss    0.208    144
Mr      0.160    412
Mrs     0.196    107
Rare    0.278     18 

              mean  count
AgeGroup                 
child        0.127     55
teen         0.208     53
young adult  0.187    284
adult        0.192    167
senior       0.125     16
missing      0.153    137 

          mean  count
IsAlone              
0        0.191    278
1        0.168    434 



,Name,Sex,Age,Pclass,actual,pred
692,"Lam, Mr. Ali",male,NaN,3,1,0
527,"Farthing, Mr. John",male,NaN,1,0,1
509,"Lang, Mr. Fang",male,26.0,3,1,0
557,"Robbins, Mr. Victor",male,NaN,1,0,1
828,"McCormack, Mr. Thomas Joseph",male,NaN,3,1,0
18,"Vander Planke, Mrs. Julius (Emelia Maria Vande...",female,31.0,3,0,1
402,"Jussila, Miss. Mari Aina",female,21.0,3,0,1
226,"Mellors, Mr. William John",male,19.0,2,1,0
220,"Sunderland, Mr. Victor Francis",male,16.0,3,1,0
430,"Bjornstrom-Steffansson, Mr. Mauritz Hakan",male,28.0,1,1,0


In [10]:
# Are errors more common when Age is missing?
print(err.groupby(err["Age"].isna())["wrong"].agg(["mean", "count"]).round(3), "\n")

# Do 3rd-class women with big families survive less?
w3 = err[(err["Sex"] == "female") & (err["Pclass"] == 3)]
print(w3.groupby(pd.cut(w3["FamilySize"], [0, 1, 3, 20]), observed=True)["actual"]
        .agg(["mean", "count"]).round(3), "\n")

# How often do men survive, by class?
print(err[err["Sex"] == "male"].groupby("Pclass")["actual"].agg(["mean", "count"]).round(3))

        mean  count
Age                
False  0.183    575
True   0.153    137 

             mean  count
FamilySize              
(0, 1]      0.627     51
(1, 3]      0.500     38
(3, 20]     0.259     27 

         mean  count
Pclass              
1       0.370     92
2       0.152     92
3       0.135    275


In [12]:
def add_features2(d):
    d = d.copy()
    d["FamilyBand"] = pd.cut(d["FamilySize"], [0, 1, 3, 20],
                             labels=["alone", "small", "large"]).astype(str)
    d["SexClass"] = d["Sex"] + "_" + d["Pclass"].astype(str)
    return d

df3 = add_features2(df2)
X3 = df3.drop(columns=["Survived", "PassengerId", "Name", "Ticket", "Cabin"])
y3 = df3["Survived"]
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, stratify=y3, random_state=42
)

cat3 = ["Sex", "Pclass", "Embarked", "Title", "FamilyBand", "SexClass"]
prep3 = ColumnTransformer([
    ("num", num_pipe, num2),
    ("cat", cat_pipe, cat3),
    ("bin", "passthrough", bin2),
])

rows = []
for name, clf in {
    "Logistic Regression + interactions": LogisticRegression(max_iter=1000),
    "Random Forest (depth 6)": RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42),
}.items():
    pipe = Pipeline([("prep", prep3), ("model", clf)])
    cv = cross_validate(pipe, X3_train, y3_train, cv=5,
                        scoring=["accuracy", "f1", "roc_auc"],
                        return_train_score=True)
    rows.append({
        "Model": name,
        "Accuracy": cv["test_accuracy"].mean(),
        "F1": cv["test_f1"].mean(),
        "Val AUC": cv["test_roc_auc"].mean(),
        "Val AUC std": cv["test_roc_auc"].std(),
        "Train AUC": cv["train_roc_auc"].mean(),
    })

pd.DataFrame(rows).set_index("Model").round(3)

,Accuracy,F1,Val AUC,Val AUC std,Train AUC
Model,,,,,
Logistic Regression + interactions,0.833,0.767,0.872,0.014,0.886
Random Forest (depth 6),0.816,0.735,0.870,0.018,0.919


In [15]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             mean_squared_error)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

# --- Classification: fit on ALL training data, score on test ONCE ---
final_clf = Pipeline([("prep", prep2),
                      ("model", LogisticRegression(max_iter=1000))])
final_clf.fit(X2_train, y2_train)

test_pred = final_clf.predict(X2_test)
test_proba = final_clf.predict_proba(X2_test)[:, 1]

print("CLASSIFICATION (test set, 179 passengers)")
print("Accuracy :", round(accuracy_score(y2_test, test_pred), 3))
print("Precision:", round(precision_score(y2_test, test_pred), 3))
print("Recall   :", round(recall_score(y2_test, test_pred), 3))
print("F1       :", round(f1_score(y2_test, test_pred), 3))
print("ROC-AUC  :", round(roc_auc_score(y2_test, test_proba), 3))
print("Baseline accuracy (always 'died'):", round(1 - y2_test.mean(), 3))
print(confusion_matrix(y2_test, test_pred), "\n")

# --- Regression: Linear Regression on known-age passengers ---
final_reg = Pipeline([("prep", prep_r), ("model", LinearRegression())])
final_reg.fit(Xr_train, yr_train)
rmse = mean_squared_error(yr_test, final_reg.predict(Xr_test)) ** 0.5

base = DummyRegressor(strategy="mean").fit(Xr_train, yr_train)
base_rmse = mean_squared_error(yr_test, base.predict(Xr_test)) ** 0.5

print(f"REGRESSION (test set, {len(yr_test)} passengers)")
print("RMSE (years):", round(rmse, 2), "| baseline RMSE:", round(base_rmse, 2))

CLASSIFICATION (test set, 179 passengers)
Accuracy : 0.832
Precision: 0.8
Recall   : 0.754
F1       : 0.776
ROC-AUC  : 0.873
Baseline accuracy (always 'died'): 0.615
[[97 13]
 [17 52]] 

REGRESSION (test set, 143 passengers)
RMSE (years): 10.45 | baseline RMSE: 13.71
